# State and Track Classification Widget Preview (Test)

This notebook contains metadata loading and rendering for `StateClassificationPanel` and `TrackClassificationPanel`.

In [1]:
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display

from behav3d.widgets.utils import PathPicker
from behav3d.widgets.metadata import MetadataLoader
from behav3d.widgets.state_classification import StateClassificationPanel
from behav3d.widgets.track_classification import TrackClassificationPanel


In [ ]:
output_dir_picker = PathPicker(
    mode='dir',
    start_dir='/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/runs/NatureBriefComm/LowDensity',
    description='Output Dir:',
)

metadata_path_picker = PathPicker(
    mode='file',
    start_dir='/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/runs/NatureBriefComm/LowDensity/behav3d',
    description='Metadata .csv:',
    filter_pattern='*.csv',
)

metadata_loader = MetadataLoader(
    metadata_path_picker=metadata_path_picker,
    output_dir_picker=output_dir_picker,
    button_description='Load Metadata',
)

display(metadata_loader)


MetadataLoader(children=(PathPicker(children=(Text(value='', description='Output Dir:', layout=Layout(flex='1 …

In [3]:
state_render_btn = widgets.Button(
    description='Render State Classification Panel',
    button_style='success',
)
track_render_btn = widgets.Button(
    description='Render Track Classification Panel',
    button_style='success',
)
state_status_html = widgets.HTML('<i>Load metadata first, then click render.</i>')
track_status_html = widgets.HTML('<i>Load metadata first, then click render.</i>')
state_panel_out = widgets.Output()
track_panel_out = widgets.Output()

def _render_state_panel(*_):
    state_panel_out.clear_output()
    with state_panel_out:
        if getattr(metadata_loader, 'metadata', None) is None:
            state_status_html.value = '<b style="color:#b00;">Load metadata first using the button above.</b>'
            print('Metadata is not loaded yet.')
            return

        state_status_html.value = '<b style="color:#080;">Rendering StateClassificationPanel...</b>'
        state_panel = StateClassificationPanel(metadata_loader=metadata_loader)
        display(state_panel.ui)

def _render_track_panel(*_):
    track_panel_out.clear_output()
    with track_panel_out:
        if getattr(metadata_loader, 'metadata', None) is None:
            track_status_html.value = '<b style="color:#b00;">Load metadata first using the button above.</b>'
            print('Metadata is not loaded yet.')
            return

        track_status_html.value = '<b style="color:#080;">Rendering TrackClassificationPanel...</b>'
        track_panel = TrackClassificationPanel(
            metadata_loader=metadata_loader,
            show_fixed_io=False,
            show_ngram_weight=False,
        )

        # Hide output-dir line for this preview.
        track_panel.output_dir_html.layout.display = 'none'

        # Add model h5ad existence indicator directly under the cell-type selector.
        model_h5ad_status = widgets.HTML('')

        def _update_model_h5ad_status(*__):
            model_path = Path(track_panel._model_adata_path())
            legacy_path = Path(track_panel._legacy_model_adata_path())
            if model_path.exists():
                model_h5ad_status.value = (
                    f"<b>Model h5ad:</b> <span style='color:#080;'>exists</span> "
                    f"({model_path})"
                )
            elif legacy_path.exists():
                model_h5ad_status.value = (
                    f"<b>Model h5ad:</b> <span style='color:#080;'>exists (legacy)</span> "
                    f"({legacy_path})"
                )
            else:
                model_h5ad_status.value = (
                    f"<b>Model h5ad:</b> <span style='color:#b00;'>missing</span> "
                    f"({model_path})"
                )

        _update_model_h5ad_status()
        track_panel.cell_type_dd.observe(_update_model_h5ad_status, names='value')

        ui_children = list(track_panel.ui.children)
        insert_idx = 2 if len(ui_children) >= 2 else len(ui_children)
        ui_children.insert(insert_idx, model_h5ad_status)
        track_panel.ui.children = tuple(ui_children)

        display(track_panel.ui)

state_render_btn.on_click(_render_state_panel)
track_render_btn.on_click(_render_track_panel)

display(
    widgets.VBox([
        widgets.HTML('<b>State Classification</b>'),
        state_status_html,
        state_render_btn,
        state_panel_out,
        widgets.HTML('<hr>'),
        widgets.HTML('<b>Track Classification</b>'),
        track_status_html,
        track_render_btn,
        track_panel_out,
    ])
)


In [4]:
_original_load = metadata_loader.load

def _wrapped_load(*args, **kwargs):
    result = _original_load(*args, **kwargs)
    if getattr(metadata_loader, 'metadata', None) is not None:
        state_status_html.value = '<b style="color:#080;">Metadata loaded. Auto-rendering state panel...</b>'
        track_status_html.value = '<b style="color:#080;">Metadata loaded. Auto-rendering track panel...</b>'
        _render_state_panel()
        _render_track_panel()
    return result

metadata_loader.load = _wrapped_load
print('Auto-render hook enabled: loading metadata now will render both panels automatically.')


Auto-render hook enabled: loading metadata now will render both panels automatically.
